# Gaia@AIP minimal example

You'll have to create an account on https://gaia.aip.de/ to use your API token later on.


This notebook: (1) builds an ADQL query from your `SOURCE_IDS`, (2) creates a TAP async job, (3) runs an SJS join to download `gaiadr3.xp_continuous_mean_spectrum` for those IDs, and (4) parses the VOTable into a pandas DataFrame.


In [13]:
# pip install requests pandas astropy pyvo

import io
import os
import time
from pathlib import Path

import requests
import pandas as pd
import pyvo as vo
from astropy.io.votable import parse_single_table


In [15]:
# CONFIG                                      
GAIA_AIP_TOKEN = os.getenv("GAIA_AIP_TOKEN", "YOUR_API_TOKEN_GOES_HERE").strip()
#                                             ^^^^^^^^^^^^^^^^^^^^^^^^^
if (not GAIA_AIP_TOKEN) or any(ord(c) > 127 for c in GAIA_AIP_TOKEN):
    raise RuntimeError("Set GAIA_AIP_TOKEN (ASCII only).")

TAP_URL = "https://gaia.aip.de/tap"
SJS_URL = "https://gaia.aip.de/uws/simple-join-service"

OUT = Path("out_gaiaxp_pvz")
OUT.mkdir(exist_ok=True)

# Source IDs to fetch (Gaia DR3 source_id)
SOURCE_IDS = [
    5853498713190525696,
]


In [16]:
# AUTH SESSION
sess = requests.Session()
sess.headers["Authorization"] = GAIA_AIP_TOKEN if GAIA_AIP_TOKEN.startswith("Token " ) else f"Token {GAIA_AIP_TOKEN}"


In [17]:
# HELPERS
def tap_create_async_job(session: requests.Session, query: str, *, lang: str = "ADQL", runid: str = "ids_for_sjs") -> str:
    url = f"{TAP_URL}/async"
    query = query.strip().rstrip(";")
    payload = {
        "REQUEST": "doQuery",
        "LANG": lang,
        "FORMAT": "votable",
        "QUERY": query,
        "RUNID": runid,
    }
    r = session.post(url, data=payload, allow_redirects=False, timeout=120)
    if r.status_code in (302, 303) and "Location" in r.headers:
        job_url = r.headers["Location"]
        if job_url.startswith("/"):
            job_url = "https://gaia.aip.de" + job_url
    else:
        raise RuntimeError(f"Unexpected TAP async response: HTTP {r.status_code}; body={r.text[:300]!r}")

    job_id = job_url.rstrip("/").split("/")[-1]
    session.post(f"{job_url}/phase", data={"PHASE": "RUN"}, timeout=60).raise_for_status()

    t0 = time.time()
    while True:
        ph = session.get(f"{job_url}/phase", timeout=60).text.strip()
        if ph in ("COMPLETED", "ERROR", "ABORTED"):
            break
        if time.time() - t0 > 180:
            raise TimeoutError("TAP async job timed out (>180s).")
        time.sleep(1.5)

    if ph != "COMPLETED":
        raise RuntimeError(f"TAP async job ended with phase={ph}")

    return job_id


def sjs_download(session: requests.Session, job_id: str, join_table: str, *,
                 column_name: str = "source_id",
                 responseformat: str = "votable",
                 data_structure: str = "COMBINED") -> Path:
    service = vo.dal.DALService(SJS_URL, session=session)
    q = service.create_query(
        job_id=job_id,
        column_name=column_name,
        responseformat=responseformat,
        join_table=join_table,
        data_structure=data_structure,
    )
    resp = q.submit(post=True)
    resp.raise_for_status()
    job = vo.io.uws.parse_job(io.BytesIO(resp.content))

    service._session.post(f"{service._baseurl}/{job.jobid}/phase", data={"PHASE": "RUN"}, stream=True).raise_for_status()

    t0 = time.time()
    while True:
        ph = service._session.get(f"{service._baseurl}/{job.jobid}/phase").text.strip()
        if ph in ("COMPLETED", "ERROR", "ABORTED"):
            break
        if time.time() - t0 > 240:
            raise TimeoutError("SJS job timed out (>240s).")
        time.sleep(1.5)

    if ph != "COMPLETED":
        raise RuntimeError(f"SJS ended with phase={ph}")

    job_url = f"{service._baseurl}/{job.jobid}"
    job2 = vo.io.uws.parse_job(io.BytesIO(service._session.get(job_url).content))

    results = job2.results
    if hasattr(results, "keys") and callable(getattr(results, "keys")):
        first_key = sorted(list(results.keys()))[0]
        href = results[first_key].href
        key = str(first_key)
    else:
        res_list = list(results)
        if not res_list:
            raise RuntimeError("SJS job has no results.")
        href = res_list[0].href
        key = "result"

    out_path = OUT / f"sjs_{job2.jobid}_{key}.vot"
    out_path.write_bytes(service._session.get(href, timeout=300).content)
    return out_path


def read_first_votable(path: Path) -> pd.DataFrame:
    tab = parse_single_table(str(path)).to_table(use_names_over_ids=True)
    return tab.to_pandas()


In [18]:
# RUN
ids_sql = ",".join(str(int(sid)) for sid in SOURCE_IDS)
q_ids_job = f"""
SELECT source_id
FROM gaiadr3.gaia_source
WHERE source_id IN ({ids_sql})
"""

tap_job_id = tap_create_async_job(sess, q_ids_job, runid="ids_for_sjs")
print("TAP job_id:", tap_job_id)

path_vot = sjs_download(
    sess,
    tap_job_id,
    "gaiadr3.xp_continuous_mean_spectrum",
    responseformat="votable",
    data_structure="COMBINED",
)
print("Saved:", path_vot)

df_xp = read_first_votable(path_vot)
print("Columns:", list(df_xp.columns)[:25], "...")
df_xp.head(2)


TAP job_id: 935c5bd2-f4c8-4ade-ae4f-1217ea3d3e7f
Saved: out_gaiaxp_pvz/sjs_faf95867-ab39-4acd-9744-e4159e09f292_result.vot
Columns: ['source_id', 'solution_id', 'bp_basis_function_id', 'bp_degrees_of_freedom', 'bp_n_parameters', 'bp_n_measurements', 'bp_n_rejected_measurements', 'bp_standard_deviation', 'bp_chi_squared', 'bp_coefficients', 'bp_coefficient_errors', 'bp_coefficient_correlations', 'bp_n_relevant_bases', 'bp_relative_shrinking', 'rp_basis_function_id', 'rp_degrees_of_freedom', 'rp_n_parameters', 'rp_n_measurements', 'rp_n_rejected_measurements', 'rp_standard_deviation', 'rp_chi_squared', 'rp_coefficients', 'rp_coefficient_errors', 'rp_coefficient_correlations', 'rp_n_relevant_bases'] ...


,source_id,solution_id,bp_basis_function_id,bp_degrees_of_freedom,bp_n_parameters,bp_n_measurements,bp_n_rejected_measurements,bp_standard_deviation,bp_chi_squared,bp_coefficients,...,rp_n_parameters,rp_n_measurements,rp_n_rejected_measurements,rp_standard_deviation,rp_chi_squared,rp_coefficients,rp_coefficient_errors,rp_coefficient_correlations,rp_n_relevant_bases,rp_relative_shrinking
0,5853498713190525696,4545469030156206080,56,4646,55,4701,19,1.122467,5853.641602,"[56947.03696840172, 13812.466168611605, 6752.4...",...,55,5694,137,1.315128,9753.001953,"[735186.2157096652, 350804.5610551923, -24394....","[284.91925, 199.77626, 246.6683, 286.87973, 25...","[0.17973025, 0.5096138, -0.258848, -0.17316186...",40,1.0
